# Beskrivelse af filtrering
Funktionen tager det rå bitemporale BBR og spytter et smukt filtreret og manipuleret datasæt ud klar til OSRM beregninger

- Koden kræver først at rækkerne alle er aktuelle (den retmæssige beskrivelse af en bygnings historiske forløb). Dvs. registreringTil.isna() og status == 6 (opført)

- Herefter ordnes temporale ændringer i koordinater og opførselsår for samme bygning_id. Logikken er simpel: En bygning kan ikke skifte opførselsår, så vi overskrive alle historiske rækker med det nyeste opførselsår. Koordinater kan principielt godt ændres, men meget lidt, og over BBR's historik er kvaliteten blevet forbedret. Derfor antages de nyeste koordinater (for hvert bygning_id) generelt at være mere retvisende. Således overskrives historiske rækker også med deres respektive nyeste koordinat. 

- Anvendelseskoder er blevet mere granulleret i BBR (fx går anvendelseskategori 320 (kontor, lager, erhverv) til en masse underkategorier fx 321 (kontor)). For alle historiske rækker med uddateret anvendelseskategori (e.g. 320) overskrives med bygningens nyeste mere præcise definition (e.g. 321). Således kan vi sortere tæthedsmålet i den senere regression til fx at være effekten af kontor m2 (eller helårsboliger, ....) på erhvervslejen.  

<a id="BBR_Bitemporalitet"></a>

## BBR Bitemporalitet

Dette er en forklaring på hvordan man sorterer i det historiske BBR datasæt, hvor alle tidligere registreringer fremgår - også dem der ikke længere er relevante.

Vi ønsker kun at beholde de observationer som stadig er "registreret" i BBR. Altså fjerne obserbationer som ikke længere er en del af datasættet "registreret". Dette er en teknikalitet hvor fx et hus registreres med virkning fra 2016 - NaT. Denne række slettes ikke når bygningens andevelse fx ændres i 2018, her vil rækken blot ikke længere være "registreret".


### Eksempel på hvordan en bygnings nye anvendelse registreres i BBR:
```text
  registreretFra    registreretTil   virkningFra    VirkningTil      id       anvendelse
     2016               NaT            2016             NaT         123          10                      bygning 123 er registreret med anvendelse 10 fra 2016 til ubestemt tid

  registreretFra    registreretTil   virkningFra    VirkningTil      id       anvendelse
     2016               2018           2016             NaT         123          10                      bygning 123 er fra 2016 til 2018 registreret som anvendelse 10 fra 2016 til ubestemt tid (forrige registrering udgår)
     2018               NaT            2016            2018         123          20                      bygning 123 står registreret fra 2018 på ubestemt tid som anvendelse 10 i perioden 2016-2018 (arkiv over historikken)
     2018               NaT            2018             NaT         123          20                      bygning 123 er nu (på ubestemt tid) registreret som anvendelse = 20
```


altså skal vi blot sorterer på registreret til = NaT (Python betingelsen: registreringTil.isna())

In [1]:
# indlæs pakker
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from pandas.errors import DtypeWarning
import warnings
warnings.filterwarnings("ignore", category=DtypeWarning)

## Definer filtreringsfunktion

In [2]:
"""Chunked BBR corrections using pandas and temporary CSV files.
Dependencies: pandas, numpy, tqdm. Uses temporary CSV files.
RAM is limited to working copies of a chunk or one unusually large building history.
"""

from pathlib import Path
from tempfile import TemporaryDirectory
import pandas as pd
from tqdm.auto import tqdm

UDFASEDE_ANVENDELSESKODER = {
    130, 210, 220, 230, 290, 310, 320, 330,
    390, 410, 420, 430, 440, 490, 520, 530,
}



def _ret_partition(frame, overwrite_coordinate, overwrite_construction_year, overwrite_main_use):
    """Apply the original rules to a whole partition using bulk operations."""
    import numpy as np

    out = frame.reset_index(drop=True).copy()
    keys = ["id_namespace", "id_lokalId"]
    valid_keys = out[keys].notna().all(axis=1)
    positions = np.flatnonzero(valid_keys.to_numpy())
    if not len(positions):
        return out

    groups, _ = pd.factorize(pd.MultiIndex.from_frame(out.loc[valid_keys, keys]), sort=False)
    repeated = np.bincount(groups)[groups] > 1
    positions = positions[repeated]
    if not len(positions):
        return out  # A building with one retained row cannot need these corrections.

    work_columns = ["virkningFra"]
    if overwrite_construction_year:
        work_columns += ["byg026Opførelsesår"]
    if overwrite_main_use:
        work_columns += ["byg021BygningensAnvendelse"]
    if overwrite_coordinate:
        work_columns += ["byg134KvalitetAfKoordinatsæt", "byg404Koordinat", "byg406Koordinatsystem"]
    work = out.iloc[positions][work_columns].copy()
    work["_pos"] = positions
    work["_group"] = groups[repeated]
    work["_time"] = pd.to_datetime(
        work["virkningFra"], utc=True, format="mixed", errors="coerce"
    ).astype("datetime64[ns, UTC]")

    # 1. Latest dated, nonmissing construction year; stable sort preserves file-order ties.
    if overwrite_construction_year:
        latest = (
            work.loc[work["_time"].notna() & work["byg026Opførelsesår"].notna()]
            .sort_values("_time", ascending=False, kind="stable")
            .drop_duplicates("_group")
            .set_index("_group")["byg026Opførelsesår"]
        )
        replacement = work["_group"].map(latest)
        use = replacement.notna()
        out.loc[work.loc[use, "_pos"], "byg026Opførelsesår"] = replacement.loc[use].to_numpy()

    # 2. Only retired use codes may be replaced; the latest date must have one unique code.
    if overwrite_main_use:
        codes = pd.to_numeric(work["byg021BygningensAnvendelse"], errors="coerce")
        last_date = work["_time"].groupby(work["_group"]).transform("max")
        latest = work["_time"].notna() & work["_time"].eq(last_date)
        latest_codes = pd.DataFrame({
            "group": work.loc[latest, "_group"],
            "code": codes.loc[latest],
        }).groupby("group")["code"].agg(["nunique", "first"])
        modern = latest_codes.loc[
            latest_codes["nunique"].eq(1)
            & ~latest_codes["first"].isin(UDFASEDE_ANVENDELSESKODER),
            "first",
        ]
        replacement = work["_group"].map(modern)
        use = codes.isin(UDFASEDE_ANVENDELSESKODER) & replacement.notna()
        out.loc[work.loc[use, "_pos"], "byg021BygningensAnvendelse"] = (
            replacement.loc[use].astype("int64").astype("string").to_numpy()
        )

    # 3. Closest strictly better ORIGINAL coordinate, using two time joins per quality.
    if overwrite_coordinate:
        quality_column = "byg134KvalitetAfKoordinatsæt"
        coordinate_column = "byg404Koordinat"
        work["_quality"] = pd.to_numeric(work[quality_column], errors="coerce").astype("float64")
        donors = work.loc[
            work["_time"].notna() & work["_quality"].notna()
            & work[coordinate_column].notna()
        ].copy()
        dated = work["_time"].notna()
        columns = [coordinate_column, quality_column, "byg406Koordinatsystem"]
        original_coordinates = out[columns].to_numpy(copy=True)

        for quality in pd.unique(work.loc[dated, "_quality"]):
            if pd.isna(quality):
                targets = work.loc[dated & work["_quality"].isna()]
                better = donors.copy()
            else:
                targets = work.loc[dated & work["_quality"].eq(quality)]
                better = donors.loc[donors["_quality"].lt(quality)].copy()
            if targets.empty or better.empty:
                continue

            # Conflicting coordinates at the same donor date are ambiguous.
            date_groups = ["_group", "_time"]
            better["_ambiguous"] = (
                better.groupby(date_groups, sort=False)[coordinate_column]
                .transform("nunique").gt(1)
            )
            best = (
                better.sort_values("_quality", kind="stable")
                .drop_duplicates(date_groups)
                .rename(columns={"_pos": "_donor_pos"})
                .sort_values("_time", kind="stable")
            )
            best["_donor_time"] = best["_time"]
            keep = date_groups + [
                "_donor_pos", "_donor_time", "_quality", "_ambiguous", coordinate_column,
            ]
            targets = targets[["_pos", "_group", "_time"]].sort_values("_time", kind="stable")
            before = pd.merge_asof(
                targets, best[keep], on="_time", by="_group", direction="backward",
            )
            after = pd.merge_asof(
                targets, best[keep], on="_time", by="_group", direction="forward",
            )
            bpos = before["_donor_pos"].fillna(-1).to_numpy(dtype="int64")
            apos = after["_donor_pos"].fillna(-1).to_numpy(dtype="int64")
            bok, aok = bpos >= 0, apos >= 0

            # Unsigned nanosecond differences avoid overflow for very old dates.
            time = targets["_time"].array.asi8.view("uint64")
            bdistance = time - before["_donor_time"].array.asi8.view("uint64")
            adistance = after["_donor_time"].array.asi8.view("uint64") - time
            bdistance[~bok] = np.iinfo("uint64").max
            adistance[~aok] = np.iinfo("uint64").max
            tie = bok & aok & (bdistance == adistance)
            same_coordinate = before[coordinate_column].eq(
                after[coordinate_column]
            ).fillna(False).to_numpy(dtype=bool)
            bambiguous = before["_ambiguous"].eq(True).to_numpy(dtype=bool)
            aambiguous = after["_ambiguous"].eq(True).to_numpy(dtype=bool)

            bquality = before["_quality"].fillna(np.inf).to_numpy(dtype=float)
            aquality = after["_quality"].fillna(np.inf).to_numpy(dtype=float)
            choose_before = bdistance < adistance
            choose_before |= tie & (
                (bquality < aquality) | ((bquality == aquality) & (bpos <= apos))
            )
            chosen = np.where(choose_before, bpos, apos)
            ambiguous = np.where(choose_before, bambiguous, aambiguous)
            accept = (
                (bok | aok) & ~ambiguous
                & ~(tie & (~same_coordinate | bambiguous | aambiguous))
            )
            target_positions = targets["_pos"].to_numpy()[accept]
            donor_positions = chosen[accept]

            # Assign all three donor fields together.
            out.loc[target_positions, columns] = original_coordinates[donor_positions]

    return out


def _append_partitions(chunk, folder, prefix, level, counts):
    """Send each building to the same temporary CSV file."""
    buckets = (chunk["_bbr_hash"] // (64 ** level)) % 64
    for bucket, part in chunk.groupby(buckets, sort=False):
        path = folder / f"{prefix}_{int(bucket):02d}.csv"
        part.to_csv(path, mode="a", header=not path.exists(), index=False)
        counts[path] = counts.get(path, 0) + len(part)


def filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder,
    input_filename,
    output_folder,
    output_filename,
    status=6,
    cols=None,
    overwrite_coordinate=True,
    overwrite_construction_year=True,
    overwrite_main_use=True,
    chunksize=100_000,
):
    """Filter and correct using pandas and temporary CSV files.

    Original retained rows supply all corrections. Missing building IDs are
    retained without correction. Output keeps the input row/column order.

    Temporary files are split until they contain at most chunksize rows,
    except when one building's history alone exceeds that limit.
    """
    if not isinstance(chunksize, int) or isinstance(chunksize, bool) or chunksize < 1:
        raise ValueError("chunksize must be a positive integer.")

    input_path = Path(input_folder) / input_filename
    output_path = Path(output_folder) / output_filename
    if input_path.resolve() == output_path.resolve():
        raise ValueError("Input and output must be different files.")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # 1. Choose output columns and the columns needed for corrections.
    available = pd.read_csv(input_path, nrows=0).columns.tolist()
    output_columns = available if cols is None else list(cols)
    corrections = any([
        overwrite_coordinate, overwrite_construction_year, overwrite_main_use
    ])
    keys = ["id_namespace", "id_lokalId"]
    required = ["registreringTil", "status"]
    if corrections:
        required += keys + ["virkningFra"]
    if overwrite_coordinate:
        required += [
            "byg134KvalitetAfKoordinatsæt",
            "byg404Koordinat",
            "byg406Koordinatsystem",
        ]
    if overwrite_construction_year:
        required += ["byg026Opførelsesår"]
    if overwrite_main_use:
        required += ["byg021BygningensAnvendelse"]

    needed = list(dict.fromkeys(output_columns + required))
    missing = sorted(set(needed) - set(available))
    if missing:
        raise ValueError(f"Missing CSV columns: {missing}")
    if len(output_columns) != len(set(output_columns)):
        raise ValueError("cols contains duplicate column names.")
    if {"_bbr_row_order", "_bbr_hash"} & set(available):
        raise ValueError("Input contains reserved internal column names.")

    part_dtypes = {column: "string" for column in needed}
    part_dtypes.update(_bbr_row_order="int64", _bbr_hash="uint64")
    rows_read = 0
    rows_written = 0

    with TemporaryDirectory(prefix="bbr_chunks_", dir=output_path.parent) as temp_folder:
        folder = Path(temp_folder)
        temporary_output = folder / "result.csv"
        pd.DataFrame(columns=output_columns).to_csv(temporary_output, index=False)

        # 2. Read/filter in chunks. Put each building's retained history
        #    into one temporary CSV, including rows from later input chunks.
        counts = {}
        with pd.read_csv(
            input_path, usecols=needed, dtype="string", chunksize=chunksize,
        ) as reader:
            reading = tqdm(reader, desc="1/3 Reading and filtering", unit="chunk")
            for chunk in reading:
                rows_read += len(chunk)
                keep = (
                    chunk["registreringTil"].isna()
                    & pd.to_numeric(chunk["status"], errors="coerce").eq(status)
                )
                filtered = chunk.loc[keep].copy()
                rows_written += len(filtered)
                reading.set_postfix(read=f"{rows_read:,}", kept=f"{rows_written:,}", refresh=False)
                if filtered.empty:
                    continue

                filtered["_bbr_row_order"] = filtered.index.astype("int64")
                if corrections:
                    filtered["_bbr_hash"] = pd.util.hash_pandas_object(
                        filtered[keys], index=False
                    )
                    # Missing IDs cannot define a building; distribute those rows individually.
                    missing_id = filtered[keys].isna().any(axis=1)
                    filtered.loc[missing_id, "_bbr_hash"] = pd.util.hash_pandas_object(
                        filtered.loc[missing_id, ["_bbr_row_order"]], index=False
                    )
                    _append_partitions(filtered, folder, "part", 0, counts)
                else:
                    for column in [
                        "byg406Koordinatsystem", "registreringFra", "virkningFra", "virkningTil"
                    ]:
                        if column in filtered:
                            filtered[column] = filtered[column].str.replace("Z", "", regex=False)
                    filtered.to_csv(
                        temporary_output, mode="a", header=False,
                        index=False, columns=output_columns,
                    )

        # 3. Correct one small temporary file at a time.
        #    If a file is too large, split it into smaller CSV files first.
        pending = [(path, count, 1) for path, count in counts.items()]
        with tqdm(total=rows_written, desc="2/3 Correcting buildings", unit="row") as progress:
            if not corrections:
                progress.update(rows_written)
            while pending:
                path, count, level = pending.pop()
                progress.set_postfix(partition_rows=f"{count:,}", remaining=len(pending), refresh=False)
                if count > chunksize:
                    if level > 10:
                        raise ValueError("Could not split a temporary file into small groups.")

                    children = {}
                    first_key = None
                    one_building = True
                    with pd.read_csv(
                        path, dtype=part_dtypes, chunksize=chunksize,
                    ) as reader:
                        for chunk in reader:
                            if first_key is None:
                                first_key = tuple(chunk[keys].iloc[0])
                            if one_building:
                                one_building = bool(
                                    chunk[keys].notna().all().all()
                                    and chunk[keys[0]].eq(first_key[0]).all()
                                    and chunk[keys[1]].eq(first_key[1]).all()
                                )
                            _append_partitions(
                                chunk, folder, path.stem, level, children
                            )
                    if not one_building:
                        pending.extend(
                            (child, size, level + 1) for child, size in children.items()
                        )
                        path.unlink()
                        continue

                    # A single building cannot be split without losing its history.
                    for child in children:
                        child.unlink()

                part = pd.read_csv(path, dtype=part_dtypes)
                part = part.sort_values("_bbr_row_order", kind="stable")
                part = _ret_partition(
                    part, overwrite_coordinate, overwrite_construction_year, overwrite_main_use
                )

                for column in [
                    "byg406Koordinatsystem", "registreringFra", "virkningFra", "virkningTil"
                ]:
                    if column in part:
                        part[column] = part[column].str.replace("Z", "", regex=False)

                # Send corrected rows back to their original input chunk.
                original_chunks = part["_bbr_row_order"] // chunksize
                for chunk_number, rows in part.groupby(original_chunks, sort=False):
                    target = folder / f"output_{int(chunk_number):09d}.csv"
                    rows.to_csv(
                        target, mode="a", header=not target.exists(), index=False,
                        columns=output_columns + ["_bbr_row_order"],
                    )
                path.unlink()
                progress.update(len(part))

        # 4. Restore original row order and write the final CSV in chunks.
        for path in tqdm(sorted(folder.glob("output_*.csv")), desc="3/3 Writing output", unit="chunk"):
            part = pd.read_csv(
                path,
                dtype={**{column: "string" for column in output_columns},
                       "_bbr_row_order": "int64"},
            ).sort_values("_bbr_row_order", kind="stable")
            part.to_csv(
                temporary_output, mode="a", header=False, index=False,
                columns=output_columns,
            )
            path.unlink()

        # Replace an existing output only after the entire run succeeds.
        temporary_output.replace(output_path)

    print(f"Read: {rows_read:,}; saved: {rows_written:,}; removed: {rows_read - rows_written:,}")
    print(f"Saved to: {output_path}")
    return output_path


c:\Users\cam\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Filtrer og manipuler BBR (udvalgte kolonner)

In [ ]:
# foldersettings
input_folder = "D:/tung_databehandling/BBR/raw" # data_folder_path_output = "P:/cmo/TFP&Tæthed/BBR/data"
input_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653.csv"
output_folder = "D:/tung_databehandling/BBR/raw"
output_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_registreringTil_isna_Status6_fixedKoordinater_fixedOpførelsesår_fixedAnvendelseskoder_selected_cols.csv"

cols_to_extract = ['id_namespace', # identifikation af et namespace inden for hvilket lokalId er unik
 'id_lokalId', # identifikation af objektet
 'kommunekode', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/Kommunekode
 'registreringFra',
 'registreringTil',
 'virkningFra', # tidspunktet hvor virkningen af den pågældende version af bygværkselementet er startet
 'virkningTil', # alle uden værdi (NaN)
 'status', # kode for bygværkselementets status i den pågældende version, dvs. elementets tilstand i den samlede livscyklus
 'byg021BygningensAnvendelse', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/BygAnvendelse
 'byg026Opførelsesår', # angiver bygningens opførselsår
 'byg027OmTilbygningsår', # Bygningens seneste om- eller tilbygningsår. Angives i formatet åååå
 'byg038SamletBygningsareal', # Det samlede bygningsareal er summen af etagernes arealer, men ekskl. kælderareal og areal af tagetage. Arealerne måles til ydersiden af ydervæggene (bruttoetageareal). Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg039BygningensSamledeBoligAreal', # Areal, der anvendes til boligformål, !!!herunder også udnyttet areal af tagetage og lovlig beboelse i kælderetage!!!. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg040BygningensSamledeErhvervsAreal', # Areal, der udelukkende anvendes til erhverv (ikke-boligformål), herunder også udnyttet areal af tagetage og kælderetage. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg041BebyggetAreal', # Arealet registreres som stueetagens areal målt til ydersiden af ydervæggene. For bygninger, hvor etagerne arealmæssigt afviger fra hinanden, angives arealet af den største etage. Kælderetagen registreres særskilt i felt ETA.20 i ''kælder''. Arealet af åbne, overdækkede terrasser, åbne altaner og åbne indgangspartier medregnes ikke. Såfremt sådanne arealer lukkes, skal arealet medregnes. Arealer af porte, luftsluser o. lign. medtages til den linie, som bygningens ydervægge iøvrigt angiver. Arealet af garage, carport eller udhus, der udgør en del af bygningen (indenfor bygningens ydervægge og under samme tag) medregnes ligeledes. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg054AntalEtager',
 #'byg070Fredning', # personligt nysgerrig hvor mange der er og hvad deres karakteristika er:) - ikke relevant for agglomerationsprojektet
 'byg134KvalitetAfKoordinatsæt', # kodeliste der angiver kvaliteten af geometrioplysninger (1: Sikker, 2(udfases): næsten sikker, 3: usikker)
 'byg404Koordinat', # Angiver bygningens geografiske repræsentation i form af et punkt
 'byg406Koordinatsystem'] # burde bruge 5 = UTM 32 Euref89 (= epsg:25832)

filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder, input_filename, output_folder, output_filename,
    status=6,
    cols=cols_to_extract,
    overwrite_coordinate=True,
    overwrite_construction_year=True,
    overwrite_main_use=True,
    chunksize=100_000,
)

## Filtrer og manipuler BBR (Samtlige dimensioner i BBR)

In [ ]:
# foldersettings
input_folder = "D:/tung_databehandling/BBR/raw" # data_folder_path_output = "P:/cmo/TFP&Tæthed/BBR/data"
input_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653.csv"
output_folder = "D:/tung_databehandling/BBR/raw"
output_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_registreringTil_isna_Status6_fixedKoordinater_fixedOpførelsesår_fixedAnvendelseskode.csv"

cols_to_extract = None

filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder, input_filename, output_folder, output_filename,
    status=6,
    cols=cols_to_extract,
    overwrite_coordinate=True,
    overwrite_construction_year=True,
    overwrite_main_use=True,
    chunksize=100_000,
)

1/3 Reading and filtering: 50chunk [02:25,  2.77s/chunk, kept=3,710,973, read=5,000,000]